In [3]:
import pandas as pd
import json
from pathlib import Path

folder = Path("OPP-115/consolidation/threshold-0.75-overlap-similarity")
csv_files = list(folder.glob("*.csv"))
print(len(csv_files))
print(csv_files[:3])

115
[WindowsPath('OPP-115/consolidation/threshold-0.75-overlap-similarity/1017_sci-news.com.csv'), WindowsPath('OPP-115/consolidation/threshold-0.75-overlap-similarity/1028_redorbit.com.csv'), WindowsPath('OPP-115/consolidation/threshold-0.75-overlap-similarity/1034_aol.com.csv')]


In [4]:
cols = ["annotation_id", "batch_id", "annotator_id", "policy_id",
        "segment_id", "category", "attributes", "date", "url"]

all_dfs=[]  #empty list to catch each files dataframe

#iterate through all the csv files and tag each with the origin file(webpage name)
for file in csv_files:
    df = pd.read_csv(file, header=None, names=cols)
    df["source_file"] = file.name
    all_dfs.append(df)

#stack all dataframes into a single new vertically alligned DF
combined_df = pd.concat(all_dfs, ignore_index=True)
#shape check
print(combined_df.shape)
print(combined_df["source_file"].nunique())
print(combined_df["category"].value_counts())

(19720, 10)
115
category
First Party Collection/Use              8149
Third Party Sharing/Collection          4817
Other                                   2306
User Choice/Control                     1677
Data Security                            699
User Access, Edit and Deletion           690
International and Specific Audiences     529
Policy Change                            463
Data Retention                           337
Do Not Track                              53
Name: count, dtype: int64


In [5]:
#extracting each sites attribute names to check for irregularities
def get_attribute_names(attrs_json):
    return set(json.loads(attrs_json).keys())

#group by category and collect all unique attribute names for each
category_attrs = {}
for category, group in combined_df.groupby("category"):
    all_names = set()
    for attrs_json in group["attributes"]:
        all_names.update(get_attribute_names(attrs_json))
    category_attrs[category] = all_names

for cat, names in category_attrs.items():
    print(cat, "->", names)

Data Retention -> {'Personal Information Type', 'Retention Period', 'Retention Purpose'}
Data Security -> {'Security Measure'}
Do Not Track -> {'Do Not Track policy'}
First Party Collection/Use -> {'User Type', 'Collection Mode', 'Does/Does Not', 'Personal Information Type', 'Purpose', 'Identifiability', 'Action First-Party', 'Choice Scope', 'Choice Type'}
International and Specific Audiences -> {'Audience Type'}
Other -> {'Other Type'}
Policy Change -> {'User Choice', 'Change Type', 'Notification Type'}
Third Party Sharing/Collection -> {'User Type', 'Action Third Party', 'Third Party Entity', 'Personal Information Type', 'Purpose', 'Identifiability', 'Does/Does Not', 'Choice Scope', 'Choice Type'}
User Access, Edit and Deletion -> {'Access Scope', 'Access Type', 'User Type'}
User Choice/Control -> {'Personal Information Type', 'Purpose', 'User Type', 'Choice Scope', 'Choice Type'}


In [ ]:
#construct new data frame skeleton
data = {
    "segment_id": [101],
    "policy_id": [102],
    "annotation_id": [213],
    "category": ["first party"], 
    "text": ["sajhajsfdfldj"], 
    "purpose": ["sfasdfkjl"], 
    "does_or_does_not": ["asdfjljsd"], 
    "entity": ["asdfla"]
}

mapped_df = pd.DataFrame(data)
mapped_df.head()

,segment_id,policy_id,annotation_id,category,text,purpose,does_or_does_not,entity
0,101,102,213,first party,sajhajsfdfldj,sfasdfkjl,asdfjljsd,asdfla


In [ ]:
records = []
for i, row in combined_df.iterrows():
    record_dict = {}
    record_dict["segment_id"] = row["segment_id"]
    record_dict["policy_id"] = row["policy_id"]
    record_dict["annotation_id"] = row["annotation_id"]
    record_dict["category"] = row["category"]

    records.append(record_dict)

df_check = pd.DataFrame(records)
print(df_check.head())
print(df_check.shape)    
# ~14% of annotation_ids have a C prefix, meaning unknown, kept as strings

   segment_id  policy_id annotation_id                    category
0           0       3905         C2760                       Other
1           0       3905         20233                       Other
2           0       3905         20234               Policy Change
3           1       3905         20324  First Party Collection/Use
4           1       3905         20325  First Party Collection/Use
(19720, 4)
